# Apheresis forecasting lab

Time-series forecasting is supervised learning **with order**.  
The extra constraint: a row at time *t* may use only information available at *t*.

This notebook uses the practice extract (`Forecasting Data for Practice v1.xlsx`).

**Order of models (on purpose)**

1. Look at trend / season / residual  
2. Difference until the series is usable  
3. Univariate statistical models (`statsmodels`)  
4. Multivariate statistical models (enrollments as a driver)  
5. How you walk into the future: recursive vs direct vs multi-output  
6. The same problem as a small ML model  

Jul–Aug 2026 are all-zero on every site (sales and enrollments). They are treated as a padded tail, not demand.


## 0. Setup


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.figsize": (10, 3.8),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

DATA = Path("/home/workdir/attachments/Forecasting Data for Practice v1.xlsx")
ORIGIN = pd.Timestamp("2026-06-01")   # last trusted actual
H = 3                                 # forecast horizon (months)


ModuleNotFoundError: No module named 'statsmodels'

In [ ]:
raw = pd.read_excel(DATA)
raw["Timeperiod"] = pd.to_datetime(raw["Timeperiod"])
raw = raw.sort_values(["unique_id", "Timeperiod"])

# trusted history only
df = raw[raw["Timeperiod"] <= ORIGIN].copy()

print(df.shape, "rows |", df.unique_id.nunique(), "sites |", df.Timeperiod.min().date(), "→", df.Timeperiod.max().date())
print("Calls all zero?", (df.Calls == 0).all())

# national series (sum of sites) — cleanest object for teaching components
nat = (
    df.groupby("Timeperiod")[["apheresis_sales", "Enrollments", "Holiday_days"]]
    .sum()
    .rename(columns={"apheresis_sales": "sales"})
    .asfreq("MS")
)
nat.tail()


## 1. Why time is the special feature

A normal ML row is exchangeable. A forecast row is not.

| If you do this | What goes wrong |
|---|---|
| Random train/test split | Future leaks into training |
| Use same-month enrollments when they are not known yet | Leakage |
| Fit on Jul–Aug 2026 zeros | You train a shutdown that did not happen |
| Score MAPE on sites that are usually 0 | Fake accuracy |

Rule used here: **features at time t must be known at the forecast origin.**


## 2. Trend, seasonality, residual

Any series can be written:

$$y_t = T_t + S_t + R_t \quad \text{(additive)}$$

or

$$y_t = T_t \times S_t \times R_t \quad \text{(multiplicative)}$$

- **Trend $T_t$**: slow level change (here: launches + growth 2022→2025).  
- **Seasonality $S_t$**: pattern that repeats every *s* steps (here *s* = 12 months).  
- **Residual $R_t$**: what the two structured pieces do not explain.

We use an additive yearly decomposition because counts are small and some months are zero. Multiplicative would divide by numbers near 0.


In [ ]:
decomp = seasonal_decompose(nat["sales"], model="additive", period=12)

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
nat["sales"].plot(ax=axes[0], title="Observed national sales")
decomp.trend.plot(ax=axes[1], title="Trend")
decomp.seasonal.plot(ax=axes[2], title="Seasonality (period = 12)")
decomp.resid.plot(ax=axes[3], title="Residual")
for ax in axes:
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

season_strength = 1 - np.nanvar(decomp.resid) / np.nanvar(decomp.seasonal + decomp.resid)
trend_strength = 1 - np.nanvar(decomp.resid) / np.nanvar(decomp.trend.dropna() + decomp.resid.reindex(decomp.trend.dropna().index))
print(f"rough season strength: {season_strength:.2f}   (1 = purely seasonal)")
print("monthly seasonal factors (mean by calendar month):")
print(decomp.seasonal.groupby(decomp.seasonal.index.month).mean().round(2).to_dict())


**How to read this on *this* file**

- Trend is the main story (volume roughly 5× from 2022 to 2025).  
- Seasonal wiggles exist but are smaller than trend. That is why “same month last year” is a weak baseline here.  
- Residual still has spikes (ID10 months of 15–23 units flowing into the national sum). Do not chase those with a 12-parameter seasonal model.


## 3. Differencing (make the series easier to model)

Many classical models want a **stationary** series: mean and variance that do not wander.

- **Lag-1 difference**: $\Delta y_t = y_t - y_{t-1}$ removes a random-walk / linear trend.  
- **Seasonal difference**: $\Delta_{12} y_t = y_t - y_{t-12}$ removes yearly level shifts.  
- **ADF test**: null = “has a unit root” (still trending). Small p-value → safer to treat as stationary.

Differencing is *not* always required. Holt–Winters and SARIMAX can carry trend/season themselves. Difference when you use ARMA, or when you want a stationary target for ML.


In [ ]:
def adf_report(s, name):
    s = s.dropna()
    stat, p, usedlag, nobs, crit, _ = adfuller(s, autolag="AIC")
    print(f"{name:28s}  ADF={stat:7.2f}  p={p:.3f}  {'stationary-ish' if p<0.05 else 'not stationary'}")

adf_report(nat["sales"], "level")
adf_report(nat["sales"].diff(), "1st difference")
adf_report(nat["sales"].diff(12), "seasonal difference s=12")
adf_report(nat["sales"].diff().diff(12), "1st + seasonal")

fig, ax = plt.subplots(1, 3, figsize=(11, 3))
nat["sales"].plot(ax=ax[0], title="Level")
nat["sales"].diff().plot(ax=ax[1], title="Δ1")
nat["sales"].diff(12).plot(ax=ax[2], title="Δ12")
plt.tight_layout()
plt.show()


## 4. Univariate forecasting (`statsmodels`)

Univariate = only the target’s own past.

We score a fixed 6-month backtest: train through Dec 2025, predict Jan–Jun 2026.


In [ ]:
BT_START = pd.Timestamp("2026-01-01")
y = nat["sales"]
y_tr, y_te = y[y.index < BT_START], y[y.index >= BT_START]

def mae(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    return float(np.mean(np.abs(a - p)))

def bias(a, p):
    return float(np.mean(np.asarray(p, float) - np.asarray(a, float)))

actual = y_te.to_numpy()
idx = y_te.index
rows = []

# --- baselines (always report these) ---
persist = np.full(len(idx), y_tr.iloc[-1])
mean12 = np.full(len(idx), y_tr.iloc[-12:].mean())
snaive = np.array([y.loc[t - pd.DateOffset(years=1)] for t in idx], float)
for name, pred in [("Persist", persist), ("Mean_12m", mean12), ("Seasonal_naive", snaive)]:
    rows.append((name, "univariate-baseline", mae(actual, pred), bias(actual, pred), pred))

# --- Holt-Winters additive, period 12 ---
hw = ExponentialSmoothing(y_tr, trend="add", seasonal="add", seasonal_periods=12).fit()
hw_pred = hw.forecast(len(idx)).to_numpy()
rows.append(("Holt_Winters_add", "univariate-stats", mae(actual, hw_pred), bias(actual, hw_pred), hw_pred))

# --- ARIMA on levels with small orders (trend handled by d=1) ---
arima = ARIMA(y_tr, order=(1, 1, 1)).fit()
arima_pred = arima.forecast(len(idx)).to_numpy()
rows.append(("ARIMA(1,1,1)", "univariate-stats", mae(actual, arima_pred), bias(actual, arima_pred), arima_pred))

# --- SARIMA: (p,d,q) x (P,D,Q,s) ---
sarima = SARIMAX(y_tr, order=(1, 1, 1), seasonal_order=(0, 1, 1, 12),
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
sarima_pred = sarima.forecast(len(idx)).to_numpy()
rows.append(("SARIMA(1,1,1)(0,1,1,12)", "univariate-stats", mae(actual, sarima_pred), bias(actual, sarima_pred), sarima_pred))

score = pd.DataFrame(rows, columns=["model", "family", "MAE", "bias", "pred"])
print(score.drop(columns="pred").to_string(index=False))

plt.figure(figsize=(10, 4))
y.plot(label="actual", color="black")
for _, r in score.iterrows():
    pd.Series(r["pred"], index=idx).plot(label=r["model"])
plt.axvline(BT_START, color="grey", ls="--", lw=1)
plt.title("National backtest — univariate")
plt.legend(loc="upper left", fontsize=8)
plt.show()


**Why each univariate family exists**

| Model | What it assumes | Why use it |
|---|---|---|
| Persist | Tomorrow = today | Dumb floor. If you lose to this, stop. |
| Mean 12m | Level is locally constant | Strong when trend recently flattened |
| Seasonal naive | $y_t = y_{t-12}$ | Strong when season >> trend |
| Holt–Winters | Level + trend + repeating season, updated by smoothing | Fast structured forecast, few knobs |
| ARIMA(p,d,q) | After *d* differences, AR + MA short memory | When residual autocorrelation is the signal |
| SARIMA(..., s=12) | ARIMA plus yearly difference / yearly MA | When you need both growth and calendar |

On *this* national series, growth dominates season, so Mean-12m / Holt often beat seasonal naive. That is a data fact, not a law.


## 5. Multivariate forecasting

Multivariate = target **plus** other series.

Here the only real driver is `Enrollments`.  
`Holiday_days` is known ahead but almost uncorrelated with sales.  
`Calls` is empty.

Two valid designs:

| Design | Features at origin | Honest? |
|---|---|---|
| Use `enroll_t` to predict `sales_t` | Only if enrollments for month *t* are already locked | Often **no** |
| Use `enroll_{t-1}`, `enroll_{t-2}` | Known once month *t-1* has closed | **Yes** |

We use lags only.


In [ ]:
def make_supervised(frame, ycol="sales"):
    x = frame.copy()
    x["y"] = x[ycol]
    x["y_l1"] = x[ycol].shift(1)
    x["e_l1"] = x["Enrollments"].shift(1)
    x["e_l2"] = x["Enrollments"].shift(2)
    x["hol"] = x["Holiday_days"]
    x["t"] = np.arange(len(x))
    return x.dropna()

sup = make_supervised(nat)
sup_tr = sup[sup.index < BT_START]
sup_te = sup[sup.index >= BT_START]

feats = ["y_l1", "e_l1", "e_l2", "hol", "t"]
Xtr = sm.add_constant(sup_tr[feats])
ols = sm.OLS(sup_tr["y"], Xtr).fit()
print(ols.summary().tables[1])


### 5.1 Three ways to produce *H* future months

You want $\hat y_{T+1},\ldots,\hat y_{T+H}$.

**Recursive (iterated)**  
Train 1-step model $\hat y_{t+1}=f(x_t)$.  
To go 3 steps: plug $\hat y_{T+1}$ back in as the next lag.  
*Cheap. Error compounds.*

**Direct**  
Train a separate model for each horizon:  
$\hat y_{t+h}=f_h(x_t)$, $h=1,2,3$.  
*No plug-in error. Needs more data. Models can disagree with each other.*

**Multi-output (one shot)**  
One model returns the whole vector $(\hat y_{t+1},\ldots,\hat y_{t+H})$.  
*Consistent path. Needs a model that supports vector targets (or stacked OLS).*


In [ ]:
# --- Recursive 1-step OLS over the backtest (uses realized lags = "teacher forcing"
#     this is evaluation with observed history, not a pure future simulation)
Xte = sm.add_constant(sup_te[feats])
ols_1step = ols.predict(Xte).to_numpy()

# --- Pure recursive from Dec 2025 origin, no future actuals ---
state = {
    "y_l1": float(y_tr.iloc[-1]),
    "e_l1": float(nat.loc[y_tr.index[-1], "Enrollments"]),
    "e_l2": float(nat.loc[y_tr.index[-2], "Enrollments"]),
    "hol": 0.0,
    "t": float(sup_tr["t"].iloc[-1]),
}
# future enrollments unknown → freeze last 3-month mean
enr_lvl = float(nat.loc[y_tr.index[-3:], "Enrollments"].mean())
rec = []
tcur = state["t"]
for ts in idx:
    tcur += 1
    row = {"const": 1.0, "y_l1": state["y_l1"], "e_l1": state["e_l1"],
           "e_l2": state["e_l2"], "hol": float(nat.loc[ts, "Holiday_days"] if ts in nat.index else 0),
           "t": tcur}
    xrow = pd.DataFrame([row], columns=list(Xtr.columns))
    yhat = float(np.asarray(ols.predict(xrow)).ravel()[0])
    yhat = max(0.0, yhat)
    rec.append(yhat)
    state["e_l2"] = state["e_l1"]
    state["e_l1"] = enr_lvl          # assumed future enrollments
    state["y_l1"] = yhat             # predicted sales becomes next lag

rec = np.array(rec)

# --- Direct: separate OLS for horizon h using only info at t ---
direct = []
base = make_supervised(nat)
for h in range(1, 7):
    tmp = base.copy()
    tmp["y_h"] = tmp["y"].shift(-h)
    trn = tmp.dropna()
    trn = trn[trn.index < BT_START]
    m = sm.OLS(trn["y_h"], sm.add_constant(trn[feats])).fit()
    # predict from the last train row (origin = Dec 2025)
    origin_row = tmp.loc[[y_tr.index[-1]], feats]
    pred_h = m.predict(sm.add_constant(origin_row, has_constant="add"))
    direct.append(float(np.asarray(pred_h).ravel()[0]))
direct = np.maximum(direct, 0)

# --- Multi-output: six stacked equations, same features at origin ---
# (already the direct collection — reported as one path)

rows2 = [
    ("OLS_1step_observed_lags", "multivariate-eval", mae(actual, ols_1step), bias(actual, ols_1step), ols_1step),
    ("OLS_recursive_frozen_enroll", "multivariate-recursive", mae(actual, rec), bias(actual, rec), rec),
    ("OLS_direct_h1to6", "multivariate-direct", mae(actual, direct), bias(actual, direct), direct),
]
score2 = pd.DataFrame(rows2, columns=["model", "family", "MAE", "bias", "pred"])
print("Univariate best so far vs multivariate walk-forward\n")
print(pd.concat([
    score.drop(columns="pred"),
    score2.drop(columns="pred"),
]).to_string(index=False))

plt.figure(figsize=(10, 4))
y_te.plot(label="actual", color="black", marker="o")
pd.Series(mean12, index=idx).plot(label="Mean_12m")
pd.Series(rec, index=idx).plot(label="OLS recursive")
pd.Series(direct, index=idx).plot(label="OLS direct")
pd.Series(ols_1step, index=idx).plot(label="OLS 1-step (observed lags)", ls="--")
plt.title("National backtest — how you step into the future")
plt.legend(fontsize=8)
plt.show()


**Read the three multivariate lines carefully**

- **1-step with observed lags** uses *actual* last month during the test window. That is a useful diagnostic of the relationship, **not** a 6-month forecast.  
- **Recursive + frozen enrollments** is the honest operational forecast.  
- **Direct** never feeds a prediction back. If it is much better than recursive, plug-in error is the problem.

If 1-step looks great and recursive looks poor, the model needs the real next enrollment — go get a plan from brand, do not add layers.


## 6. Same problem as a small ML model

Once you have a supervised table `(X_t → y_{t+h})`, any regressor can sit where OLS sat.

We stay with **ridge** implemented in plain numpy so the notebook has no extra dependency. Ridge = linear model with an L2 penalty: shrink noisy lags instead of dropping them.

$$
\hat\beta = (X^\top X + \lambda I)^{-1} X^\top y
$$

Still **time-ordered**. Still **no shuffle**.


In [ ]:
def ridge_fit(X, y, lam=5.0):
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    n = X.shape[1]
    A = X.T @ X
    A[np.diag_indices(n)] += lam
    # do not penalize intercept (column 0)
    A[0, 0] -= lam
    return np.linalg.solve(A, X.T @ y)

def ridge_predict(X, beta):
    return np.asarray(X, float) @ beta

Xtr_np = sm.add_constant(sup_tr[feats]).to_numpy(float)
ytr_np = sup_tr["y"].to_numpy(float)
beta = ridge_fit(Xtr_np, ytr_np, lam=8.0)
print("ridge coefs [const, y_l1, e_l1, e_l2, hol, t]:")
print(np.round(beta, 3))

# recursive ridge, same frozen-enrollment protocol as OLS
state = {
    "y_l1": float(y_tr.iloc[-1]),
    "e_l1": float(nat.loc[y_tr.index[-1], "Enrollments"]),
    "e_l2": float(nat.loc[y_tr.index[-2], "Enrollments"]),
}
enr_lvl = float(nat.loc[y_tr.index[-3:], "Enrollments"].mean())
tcur = float(sup_tr["t"].iloc[-1])
ridge_rec = []
for ts in idx:
    tcur += 1
    x = np.array([1.0, state["y_l1"], state["e_l1"], state["e_l2"],
                  float(nat.loc[ts, "Holiday_days"] if ts in nat.index else 0), tcur])
    yhat = max(0.0, float(x @ beta))
    ridge_rec.append(yhat)
    state["e_l2"] = state["e_l1"]
    state["e_l1"] = enr_lvl
    state["y_l1"] = yhat
ridge_rec = np.array(ridge_rec)

print(f"Ridge recursive MAE={mae(actual, ridge_rec):.2f}  bias={bias(actual, ridge_rec):.2f}")
print(f"OLS   recursive MAE={mae(actual, rec):.2f}  bias={bias(actual, rec):.2f}")
print(f"Mean12            MAE={mae(actual, mean12):.2f}  bias={bias(actual, mean12):.2f}")


Tree / boosting / neural nets plug into the **same** `X → y_{t+h}` table.

They do not get a free pass on leakage. A LightGBM trained on shuffled months is not a forecast model.

For 10 short site series, start linear. Use trees only after the linear recursive forecast loses on **volume-weighted MAE** at the big sites (ID10, ID7, ID1).


## 7. Site-level production forecast (Jul–Sep 2026)

National was the classroom. Operations needs sites.

- Mature / mid (ID1, ID4, ID5, ID7, ID10): 50/50 **Mean-12m + recursive OLS** (lags of sales and enrollments).  
- Intermittent / late start (ID2, ID3, ID6, ID8, ID9): Croston-style $P(\text{sale})\times$ typical size.  

Why split: 75% of historical units sit in three mature sites. Predicting 0 on ID6 is cheap MAE and the wrong decision.


In [ ]:
MATURE_MID = ["ID1", "ID4", "ID5", "ID7", "ID10"]
INTER = ["ID2", "ID3", "ID6", "ID8", "ID9"]
HORIZON = pd.date_range(ORIGIN + pd.offsets.MonthBegin(1), periods=H, freq="MS")

# holiday calendar from last year (known ahead)
holi = (
    raw.drop_duplicates("Timeperiod")
    .set_index("Timeperiod")["Holiday_days"]
)

def site_frame(uid):
    g = df[df.unique_id == uid].set_index("Timeperiod")[["apheresis_sales", "Enrollments", "Holiday_days"]].asfreq("MS")
    g = g.rename(columns={"apheresis_sales": "sales"})
    return g

# pooled OLS on mature+mid only
parts = []
for uid in MATURE_MID:
    g = site_frame(uid).copy()
    g["uid"] = uid
    g["y_l1"] = g["sales"].shift(1)
    g["e_l1"] = g["Enrollments"].shift(1)
    g["e_l2"] = g["Enrollments"].shift(2)
    g["t"] = np.arange(len(g))
    parts.append(g)
panel = pd.concat(parts).dropna()
dummies = pd.get_dummies(panel["uid"], prefix="id", drop_first=True, dtype=float)
Xp = sm.add_constant(pd.concat([panel[["y_l1", "e_l1", "e_l2", "t", "Holiday_days"]].astype(float), dummies], axis=1))
ols_site = sm.OLS(panel["sales"].astype(float), Xp).fit()

def recursive_site(uid, model, cols):
    g = site_frame(uid).copy()
    g["t"] = np.arange(len(g))
    last = g.iloc[-1]
    enr_lvl = float(g["Enrollments"].iloc[-3:].mean())
    state = {
        "y_l1": float(last["sales"]),
        "e_l1": float(last["Enrollments"]),
        "e_l2": float(g["Enrollments"].iloc[-2]),
        "t": float(last["t"]),
    }
    out = []
    for ts in HORIZON:
        state["t"] += 1
        hol = float(holi.get(ts - pd.DateOffset(years=1), 0))
        row = {c: 0.0 for c in cols}
        row["const"] = 1.0
        row["y_l1"] = state["y_l1"]
        row["e_l1"] = state["e_l1"]
        row["e_l2"] = state["e_l2"]
        row["t"] = state["t"]
        row["Holiday_days"] = hol
        key = f"id_{uid}"
        if key in row:
            row[key] = 1.0
        yhat = max(0.0, float(model.predict(np.array([row[c] for c in cols]).reshape(1, -1))[0]))
        out.append(yhat)
        state["e_l2"] = state["e_l1"]
        state["e_l1"] = enr_lvl
        state["y_l1"] = yhat
    return np.array(out)

def croston_like(s, n):
    w = s.iloc[-min(18, len(s)):]
    p = float((w > 0).mean())
    pos = w[w > 0]
    size = float(pos.mean()) if len(pos) else 0.0
    return np.full(n, p * size)

rows = []
for uid, g0 in df.groupby("unique_id"):
    g = g0.set_index("Timeperiod")["apheresis_sales"]
    mean12 = float(g.iloc[-12:].mean())
    if uid in MATURE_MID:
        ols_p = recursive_site(uid, ols_site, list(Xp.columns))
        chosen = 0.5 * mean12 + 0.5 * ols_p
        method = "blend_mean12_OLS"
    else:
        chosen = croston_like(g, H)
        method = "intermittent"
    chosen = np.clip(chosen, 0, None)
    for ts, val in zip(HORIZON, chosen):
        rows.append({
            "unique_id": uid,
            "month": ts.strftime("%Y-%m"),
            "method": method,
            "forecast": round(float(val), 2),
            "forecast_units": int(round(float(val))),
        })

fcst = pd.DataFrame(rows)
nat_fcst = fcst.groupby("month", as_index=False).agg(forecast=("forecast", "sum"), units=("forecast_units", "sum"))
print("Site forecast")
print(fcst.pivot(index="unique_id", columns="month", values="forecast_units").to_string())
print("\nNational")
print(nat_fcst.to_string(index=False))


## 8. What to take back to the team

1. Forecasting is prediction **without shuffling time**.  
2. Decompose first: on this file **trend > season**.  
3. Difference when you need stationarity; Holt–Winters / SARIMAX can carry trend themselves.  
4. Univariate models are the baseline. Multivariate only helps if the extra series will exist at origin.  
5. Recursive vs direct vs multi-output is about *how you walk H steps*, not about which library you import.  
6. An ML model is the same supervised table with a different $f$. It does not cancel leakage.  
7. Score **MAE and bias, volume-weighted**. Do not use MAPE on 0–2 unit sites.  
8. Production number lives in `Apheresis_Forecast_Solution.xlsx` (Jul–Sep 2026, origin June 2026).

If June 2026 reporting is incomplete, change `ORIGIN` and rerun this notebook.
